In [1]:
import os

os.environ['HF_ENDPOINT'] = 'http://hf-mirror.com'
# 下列两个任选一个 Windows的时候，自行修改 模型保存的home路径
#os.environ['TRANSFORMERS_CACHE'] = '/opt/work/huggingface/hub'
# os.environ['XDG_CACHE_HOME'] = '/opt/work'

In [2]:
from datasets import load_dataset
import torch
import torch.nn as nn
import numpy as np

from dataclasses import dataclass
from typing import Union, Tuple, Optional

import torch
from torch import nn
from transformers import BertPreTrainedModel, BertModel, BertConfig, BertTokenizer
from transformers.utils import ModelOutput

D:\anaconda3\envs\cpu_default\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
import json

# 模型迁移训练

## schema映射mapping信息加载

In [5]:
schema_path = "./duie2.0_original/duie_schema.json"
predicate2id = {}
predicate2entity = {}
with open(schema_path, "r", encoding="utf-8") as reader:
    for line in reader:
        line = json.loads(line)
        for key,value in line['object_type'].items():
            predicate = line['predicate'] + "_" + key
            predicate2id[predicate] = len(predicate2id)
            predicate2entity[predicate] = (line['subject_type'], value)
id2predicate = {_v:_k for _k,_v in predicate2id.items()}
print(f"关系和id的映射mapping:\n{len(predicate2id)}\n{predicate2id}\n")
print(f"关系对应的主客体类型映射mapping:\n{len(predicate2entity)}\n{predicate2entity}\n")

关系和id的映射mapping:
55
{'毕业院校_@value': 0, '嘉宾_@value': 1, '配音_inWork': 2, '配音_@value': 3, '主题曲_@value': 4, '代言人_@value': 5, '所属专辑_@value': 6, '父亲_@value': 7, '作者_@value': 8, '上映时间_inArea': 9, '上映时间_@value': 10, '母亲_@value': 11, '专业代码_@value': 12, '占地面积_@value': 13, '邮政编码_@value': 14, '票房_inArea': 15, '票房_@value': 16, '注册资本_@value': 17, '主角_@value': 18, '妻子_@value': 19, '编剧_@value': 20, '气候_@value': 21, '歌手_@value': 22, '获奖_inWork': 23, '获奖_onDate': 24, '获奖_@value': 25, '获奖_period': 26, '校长_@value': 27, '创始人_@value': 28, '首都_@value': 29, '丈夫_@value': 30, '朝代_@value': 31, '饰演_inWork': 32, '饰演_@value': 33, '面积_@value': 34, '总部地点_@value': 35, '祖籍_@value': 36, '人口数量_@value': 37, '制片人_@value': 38, '修业年限_@value': 39, '所在城市_@value': 40, '董事长_@value': 41, '作词_@value': 42, '改编自_@value': 43, '出品公司_@value': 44, '导演_@value': 45, '作曲_@value': 46, '主演_@value': 47, '主持人_@value': 48, '成立日期_@value': 49, '简称_@value': 50, '海拔_@value': 51, '号_@value': 52, '国籍_@value': 53, '官方语言_@value': 54}

关系对应的主客体类型映射mappi

In [6]:
REL_NUM = len(predicate2id) # relationship 关系数目
PREDICATE2ID = predicate2id # 关系和id之间的映射mapping

## 加载原始数据

In [7]:
dataset = load_dataset(
    "text",
    data_dir="./duie2.0_original",
    data_files={
        "train": "duie_train_min.json",
        "test": "duie_dev_min.json"
        # "train": "duie_train.json",
        # "test": "duie_dev.json"
    }
)
print(type(dataset))
print(dataset)
print(dataset['train'][0])

Generating train split: 9 examples [00:00, 289.65 examples/s]
Generating test split: 18 examples [00:00, 1755.80 examples/s]

<class 'datasets.dataset_dict.DatasetDict'>
DatasetDict({
    train: Dataset({
        features: ['text'],
        num_rows: 9
    })
    test: Dataset({
        features: ['text'],
        num_rows: 18
    })
})
{'text': '{"text": "《邪少兵王》是冰火未央写的网络小说连载于旗峰天下", "spo_list": [{"predicate": "作者", "object_type": {"@value": "人物"}, "subject_type": "图书作品", "object": {"@value": "冰火未央"}, "subject": "邪少兵王"}]}'}


In [10]:
# 全量数据抽样10% / 0.5% 以及数据分割
from datasets import DatasetDict

dataset = DatasetDict({
    k: ds.take(int(len(ds) * 0.005))
    for k, ds in dataset.shuffle(24).items()
})
dataset

DatasetDict({
    train: Dataset({
        features: ['text'],
        num_rows: 855
    })
    test: Dataset({
        features: ['text'],
        num_rows: 103
    })
})

## 分词器恢复

In [8]:
tokenizer = BertTokenizer.from_pretrained("bert-base-chinese")
tokenizer

BertTokenizer(name_or_path='bert-base-chinese', vocab_size=21128, model_max_length=512, is_fast=False, padding_side='right', truncation_side='right', special_tokens={'unk_token': '[UNK]', 'sep_token': '[SEP]', 'pad_token': '[PAD]', 'cls_token': '[CLS]', 'mask_token': '[MASK]'}, clean_up_tokenization_spaces=True),  added_tokens_decoder={
	0: AddedToken("[PAD]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	100: AddedToken("[UNK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	101: AddedToken("[CLS]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	102: AddedToken("[SEP]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	103: AddedToken("[MASK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
}

## 数据转换处理
- 分词、tokenid转换、构建数据

In [9]:
def search(pattern, sequence):
    """
    从sequence中找到和pattern相同的子序列
    """
    n = len(pattern)
    for i in range(len(sequence)):
        if sequence[i:i + n] == pattern:
            return i
    return -1

In [11]:
def record_process_fn(example):
    obj = json.loads(example['text']) # 将文本数据转换为json格式
    text = obj['text'] # 原始文本
    spo_list = obj['spo_list'] # 标注数据
    
    # 将原始文本进行分词、tokenid转换: attention_mask + input_ids
    item = tokenizer(
        text, 
        truncation=True,
        return_token_type_ids=False, 
        return_attention_mask=True,
        return_length=True
    )
    input_ids = item['input_ids']
    attention_mask = item['attention_mask']
    t = item['length'] # 序列长度

    # 构造映射标签
    labels = {} # 存储 主体片段 -> (关系，客体片段)
    for spo in spo_list:
        sub_span = spo['subject']
        if sub_span not in labels:
            labels[sub_span] = []
        predicate = spo['predicate']
        for predicate_suffix, obj_span in spo['object'].items():
            labels[sub_span].append((predicate + "_" + predicate_suffix,obj_span))
    
    # 构建模型映射mapping
    m = len(labels)
    sub_label = np.zeros((m, t, 2))
    sub_mask = np.zeros((m, t))
    obj_label = np.zeros((m, t, 2 * REL_NUM))
    for sub_idx, sub_span in enumerate(labels.keys()):
        # 主体标注
        cur_sub_ids = tokenizer(sub_span, add_special_tokens=False)['input_ids']
        cur_sub_start_idx = search(cur_sub_ids, input_ids) # 子序列位置查询
        if cur_sub_start_idx == -1:
            raise ValueError(example) # 异常抛出
        cur_sub_end_idx = cur_sub_start_idx + len(cur_sub_ids) - 1
        sub_label[sub_idx][cur_sub_start_idx][0] = 1 # 主体头对应label标记为1
        sub_label[sub_idx][cur_sub_end_idx][1] = 1 # 主体尾对应label标记为1
        sub_mask[sub_idx][cur_sub_start_idx:cur_sub_end_idx+1] = 1 # 实体对应范围标记为1
        
        # 客体关系标注
        for p,obj_span in labels[sub_span]:
            cur_obj_ids = tokenizer(obj_span, add_special_tokens=False)['input_ids']
            cur_obj_start_idx = search(cur_obj_ids, input_ids) # 头位置
            if cur_obj_start_idx == -1:
                raise ValueError(example) # 异常抛出
            cur_obj_end_idx = cur_obj_start_idx + len(cur_obj_ids) - 1 # 尾位置
            
            pid = PREDICATE2ID[p] # 对应关系id
            obj_label[sub_idx][cur_obj_start_idx][2 * pid] = 1 # 头标记为1
            obj_label[sub_idx][cur_obj_end_idx][2 * pid + 1] = 1 # 尾标记为1
    
    return {
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "sub_labels": sub_label,
        "sub_masks": sub_mask,
        "obj_labels": obj_label
    }

In [12]:
def record_process_fn_with_exception(example):
    try:
        return record_process_fn(example)
    except Exception as e:
        import logging
        
        logging.error("操作异常", exc_info=e)
        return {
            'input_ids': None,
            'attention_mask': None,
            'sub_labels': None,
            'sub_masks': None,
            'obj_labels': None
        }

In [13]:
dataset = dataset.map(
    record_process_fn_with_exception,
    batched=False # 不能够批次处理
)
dataset

Map: 100%|██████████| 18/18 [00:00<00:00, 619.61 examples/s]


DatasetDict({
    train: Dataset({
        features: ['text', 'input_ids', 'attention_mask', 'sub_labels', 'sub_masks', 'obj_labels'],
        num_rows: 9
    })
    test: Dataset({
        features: ['text', 'input_ids', 'attention_mask', 'sub_labels', 'sub_masks', 'obj_labels'],
        num_rows: 18
    })
})

In [14]:
def error_record_filter(example):
    return example['input_ids'] is not None

dataset = dataset.filter(error_record_filter)
dataset

Filter: 100%|██████████| 18/18 [00:00<00:00, 136.28 examples/s]


DatasetDict({
    train: Dataset({
        features: ['text', 'input_ids', 'attention_mask', 'sub_labels', 'sub_masks', 'obj_labels'],
        num_rows: 9
    })
    test: Dataset({
        features: ['text', 'input_ids', 'attention_mask', 'sub_labels', 'sub_masks', 'obj_labels'],
        num_rows: 18
    })
})

In [15]:
dataset['train'][0]

{'text': '{"text": "《邪少兵王》是冰火未央写的网络小说连载于旗峰天下", "spo_list": [{"predicate": "作者", "object_type": {"@value": "人物"}, "subject_type": "图书作品", "object": {"@value": "冰火未央"}, "subject": "邪少兵王"}]}',
 'input_ids': [101,
  517,
  6932,
  2208,
  1070,
  4374,
  518,
  3221,
  1102,
  4125,
  3313,
  1925,
  1091,
  4638,
  5381,
  5317,
  2207,
  6432,
  6825,
  6770,
  754,
  3186,
  2292,
  1921,
  678,
  102],
 'attention_mask': [1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1],
 'sub_labels': [[[0.0, 0.0],
   [0.0, 0.0],
   [1.0, 0.0],
   [0.0, 0.0],
   [0.0, 0.0],
   [0.0, 1.0],
   [0.0, 0.0],
   [0.0, 0.0],
   [0.0, 0.0],
   [0.0, 0.0],
   [0.0, 0.0],
   [0.0, 0.0],
   [0.0, 0.0],
   [0.0, 0.0],
   [0.0, 0.0],
   [0.0, 0.0],
   [0.0, 0.0],
   [0.0, 0.0],
   [0.0, 0.0],
   [0.0, 0.0],
   [0.0, 0.0],
   [0.0, 0.0],
   [0.0, 0.0],
   [0.0, 0.0],
   [0.0, 0.0],
   [0.0, 0.0]]],
 'sub_masks': [[0.0,
   0.0,
   1.0,
  

## 创建模型

In [16]:
@dataclass
class EntityRelationPointerClassifierOutput(ModelOutput):
    loss: Optional[torch.FloatTensor] = None
    sub_loss: Optional[torch.FloatTensor] = None
    obj_loss: Optional[torch.FloatTensor] = None
    sub_logits: torch.FloatTensor = None
    obj_logits: torch.FloatTensor = None


class PointerBertModel(BertPreTrainedModel):
    def __init__(self, config):
        super().__init__(config)
        self.num_labels = config.num_labels  # 标签数目直接作为关系数目

        # 如果模型是基于transformers库自定义的模型，那么不建议写成BertModel.from_pretrained, 属性名称必须为bert
        self.bert = BertModel(config, add_pooling_layer=False)
        classifier_dropout = (
            config.classifier_dropout if config.classifier_dropout is not None else config.hidden_dropout_prob
        )
        self.dropout = nn.Dropout(classifier_dropout)
        self.fc_features = nn.Sequential(
            nn.Linear(config.hidden_size, config.hidden_size),
            nn.Dropout(classifier_dropout),
            nn.ReLU()
        )
        self.sub_classify = nn.Linear(config.hidden_size, 2 * 1)
        self.obj_classify = nn.Linear(config.hidden_size, 2 * self.num_labels)

        # Initialize weights and apply final processing
        self.post_init()

    def forward(
            self,
            input_ids: Optional[torch.Tensor] = None,
            attention_mask: Optional[torch.Tensor] = None,
            sub_labels: Optional[torch.Tensor] = None,
            sub_masks: Optional[torch.Tensor] = None,
            obj_labels: Optional[torch.Tensor] = None,
            return_dict: Optional[bool] = None
    ) -> Union[Tuple[torch.Tensor], EntityRelationPointerClassifierOutput]:
        """
        :param input_ids: [bs,t]
        :param attention_mask: [bs,t]
        :param sub_labels: 主体标签(头尾) [bs,m,t,2] bs个样本，每个样本m个主体，每个主体对应的t个token上的头尾标签，填充标签使用-100
        :param sub_masks: 主体标签对应的mask矩阵 [bs,m,t] 针对每个主体来讲，属于主体的token为1，否则为0
        :param obj_labels: 客体关系标签(头尾) [bs,m,t,2*rel_num] bs个样本, 每个样本m个主体，每个主体对应t个token的客体关系头尾标签
        :param return_dict: 是否返回dict字典格式
        :return:
        """
        # 调用bert模型获取bert输出特征向量
        outputs = self.bert(input_ids, attention_mask=attention_mask, return_dict=False, output_hidden_states=True)

        # 提取最后一层的特征向量
        sequence_output = outputs[0]  # 最后一层的输出
        sequence_output = self.dropout(sequence_output)
        sequence_output = self.fc_features(sequence_output)  # [bs,t,e]
        expand_sequence_output = sequence_output.unsqueeze(dim=1)  # [bs,1,t,e]

        # 针对主体进行计算
        sub_logits = self.sub_classify(sequence_output)  # [bs,t,2] 每个样本、每个token是否属于头尾的置信度

        # 提取主体对应特征
        sub_feature_masks = sub_masks.unsqueeze(dim=-1).to(dtype=sequence_output.dtype)  # [bs,m,t] -> [bs,m,t,1]
        # [bs,m,t,1] * [bs,1,t,e] --> [bs,m,t,e]
        sub_features = sub_feature_masks * expand_sequence_output
        # [bs,m,1,e] 针对每个主体提取对应的特征向量(对应范围内的token特征向量均值)
        sub_features = sub_features.sum(dim=2, keepdim=True) / (sub_feature_masks.sum(dim=2, keepdim=True) + 1e-8)

        # 合并得到客体判断特征向量
        # [bs,m,1,e] + [bs,1,t,e] --> [bs,m,t,e]
        obj_features = expand_sequence_output + sub_features

        # 客体全连接判断 [bs,m,t,e] * [e,2*rel_num] -> [bs,m,t,2*rel_num]
        obj_logits = self.obj_classify(obj_features)

        sub_loss: Optional[torch.Tensor] = None
        if sub_labels is not None:
            bs, m, t, _ = sub_labels.shape
            sub_loss_logits = sub_logits.unsqueeze(dim=1)  # [bs,1,t,2]
            sub_loss_logits = torch.tile(sub_loss_logits, dims=(1, m, 1, 1))  # [bs,m,t,2]
            # sigmoid交叉熵损失函数
            sub_loss_fn = nn.BCEWithLogitsLoss(reduction='none')
            sub_loss = sub_loss_fn(sub_loss_logits, sub_labels.to(dtype=sub_loss_logits.dtype))  # labels填充值为-100

            # 头尾标签位置损失
            pos_labels = sub_labels == 1
            pos_label_num = pos_labels.sum()
            pos_sub_loss = sub_loss * pos_labels
            pos_sub_loss = pos_sub_loss.sum() / (pos_label_num + 1e-8)

            # 不属于头尾标签位置损失
            neg_labels = sub_labels == 0
            neg_label_num = neg_labels.sum()
            neg_sub_loss = (sub_loss * neg_labels).view(-1)
            neg_sub_loss = torch.topk(
                neg_sub_loss, k=min(pos_label_num.item() * 5, neg_label_num.item())
            ).values
            neg_sub_loss = neg_sub_loss.mean()

            # 合并正负损失
            sub_loss = pos_sub_loss + neg_sub_loss

        obj_loss: Optional[torch.Tensor] = None
        if obj_labels is not None:
            # sigmoid交叉熵损失函数
            obj_loss_fn = nn.BCEWithLogitsLoss(reduction='none')
            # bs个样本，每个样本m个主体，每个主体对应t个token，每个token属于?个类别的置信度
            obj_loss = obj_loss_fn(obj_logits, obj_labels.to(dtype=obj_logits.dtype))  # [bs,m,t,2*rel_num]

            # 头尾标签位置损失
            pos_obj_labels = obj_labels == 1
            pos_obj_label_num = pos_obj_labels.sum()
            pos_obj_loss = obj_loss * pos_obj_labels
            pos_obj_loss = pos_obj_loss.sum() / (pos_obj_label_num + 1e-8)

            # 不属于头尾标签位置损失
            neg_obj_labels = obj_labels == 0
            neg_obj_label_num = neg_obj_labels.sum()
            neg_obj_loss = (obj_loss * neg_obj_labels).view(-1)
            neg_obj_loss = torch.topk(
                neg_obj_loss,
                k=min(pos_obj_label_num.item() * 5, neg_obj_label_num.item())
            ).values
            neg_obj_loss = neg_obj_loss.mean()

            obj_loss = pos_obj_loss + neg_obj_loss

        loss: Optional[torch.Tensor] = None
        if sub_loss is not None and obj_loss is not None:
            loss = sub_loss + obj_loss
        elif sub_loss is not None:
            loss = sub_loss
        elif obj_loss is not None:
            loss = obj_loss

        if not return_dict:
            output = (sub_logits, obj_logits)
            # noinspection PyTypeChecker
            return ((loss, sub_loss, obj_loss) + output) if loss is not None else output

        return EntityRelationPointerClassifierOutput(
            loss=loss,
            sub_loss=sub_loss,
            obj_loss=obj_loss,
            sub_logits=sub_logits,
            obj_logits=obj_logits
        )

In [17]:

def tt01():
    cfg = BertConfig(
        vocab_size=1000,
        hidden_size=128,
        num_hidden_layers=2,
        num_attention_heads=2,
        intermediate_size=128 * 4,
        num_labels=3  # 关系数目
    )
    net = PointerBertModel(cfg)
    print(net)

    # 1个样本，每个样本10个token，有填充
    input_ids = torch.tensor([
        [1, 2, 3, 4, 5, 6, 7, 8, 9, 10],
        [11, 12, 13, 14, 15, 16, 17, 0, 0, 0]
    ])
    attention_mask = torch.tensor([
        [1, 1, 1, 1, 1, 1, 1, 1, 1, 1],
        [1, 1, 1, 1, 1, 1, 1, 0, 0, 0]
    ])
    # 1个样本，第一个样本有2个主体，第二个样本有一个主体
    sub_labels = torch.tensor([
        [
            [
                [0, 0],  # 第一个token
                [1, 0],
                [0, 1],
                [0, 0],
                [0, 0],
                [0, 0],
                [0, 0],
                [0, 0],
                [0, 0],
                [0, 0]
            ],  # 第一个样本、第一个主体
            [
                [0, 0],  # 第一个token
                [0, 0],
                [0, 0],
                [0, 0],
                [1, 0],
                [0, 0],
                [0, 0],
                [0, 1],
                [0, 0],
                [0, 0]
            ]
        ],  # 第一个样本
        [
            [
                [0, 0],  # 第一个token
                [0, 0],
                [1, 1],
                [0, 0],
                [0, 0],
                [0, 0],
                [0, 0],
                [-100, -100],
                [-100, -100],
                [-100, -100]
            ],
            [
                [-100, -100],  # 第一个token
                [-100, -100],
                [-100, -100],
                [-100, -100],
                [-100, -100],
                [-100, -100],
                [-100, -100],
                [-100, -100],
                [-100, -100],
                [-100, -100]
            ]
        ]
    ])
    sub_masks = torch.tensor([
        [
            [0, 1, 1, 0, 0, 0, 0, 0, 0, 0],
            [0, 0, 0, 0, 1, 1, 1, 1, 0, 0]
        ],
        [
            [0, 0, 1, 0, 0, 0, 0, 0, 0, 0],
            [0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
        ]
    ])
    obj_labels = torch.tensor([
        [
            [
                [0, 0, 0, 0, 0, 0],  # 第一个token
                [0, 0, 1, 1, 0, 0],
                [0, 0, 0, 0, 0, 0],
                [1, 0, 0, 0, 0, 0],
                [0, 0, 0, 0, 0, 0],
                [0, 1, 0, 0, 0, 0],
                [0, 0, 0, 0, 0, 0],
                [1, 0, 0, 0, 0, 0],
                [0, 1, 0, 0, 0, 0],
                [0, 0, 0, 0, 0, 0]
            ],  # 第一个样本、第一个主体
            [
                [0, 0, 0, 0, 0, 0],
                [0, 0, 0, 0, 0, 0],
                [0, 0, 0, 0, 1, 0],
                [0, 0, 0, 0, 0, 0],
                [0, 0, 0, 0, 0, 1],
                [0, 0, 0, 0, 0, 0],
                [0, 0, 0, 0, 0, 0],
                [0, 0, 0, 0, 0, 0],
                [0, 0, 0, 0, 0, 0],
                [0, 0, 0, 0, 0, 0]
            ]
        ],  # 第一个样本
        [
            [
                [0, 0, 0, 0, 0, 0],
                [0, 0, 1, 0, 0, 0],
                [0, 0, 0, 0, 0, 0],
                [0, 0, 0, 1, 0, 0],
                [1, 0, 0, 0, 0, 0],
                [0, 1, 0, 0, 0, 0],
                [0, 0, 0, 0, 0, 0],
                [-100, -100, -100, -100, -100, -100],
                [-100, -100, -100, -100, -100, -100],
                [-100, -100, -100, -100, -100, -100]
            ],
            [
                [-100, -100, -100, -100, -100, -100],
                [-100, -100, -100, -100, -100, -100],
                [-100, -100, -100, -100, -100, -100],
                [-100, -100, -100, -100, -100, -100],
                [-100, -100, -100, -100, -100, -100],
                [-100, -100, -100, -100, -100, -100],
                [-100, -100, -100, -100, -100, -100],
                [-100, -100, -100, -100, -100, -100],
                [-100, -100, -100, -100, -100, -100],
                [-100, -100, -100, -100, -100, -100]
            ]
        ]
    ])

    r = net(
        input_ids=input_ids,
        attention_mask=attention_mask,
        sub_labels=sub_labels,
        sub_masks=sub_masks,
        obj_labels=obj_labels,
    )
    print(r)

tt01() # 测试模型

PointerBertModel(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(1000, 128, padding_idx=0)
      (position_embeddings): Embedding(512, 128)
      (token_type_embeddings): Embedding(2, 128)
      (LayerNorm): LayerNorm((128,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-1): 2 x BertLayer(
          (attention): BertAttention(
            (self): BertSdpaSelfAttention(
              (query): Linear(in_features=128, out_features=128, bias=True)
              (key): Linear(in_features=128, out_features=128, bias=True)
              (value): Linear(in_features=128, out_features=128, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=128, out_features=128, bias=True)
              (LayerNorm): LayerNorm((128,), eps=1e-12, elementwise

In [18]:
# 部分参数迁移
model = PointerBertModel.from_pretrained(
    "bert-base-chinese",
    num_labels=REL_NUM # 重置标签数目 --> 设置为关系的类别数
)
model.config.rel2id=dict(PREDICATE2ID)
model

A parameter name that contains `beta` will be renamed internally to `bias`. Please use a different name to suppress this warning.
A parameter name that contains `gamma` will be renamed internally to `weight`. Please use a different name to suppress this warning.
A parameter name that contains `beta` will be renamed internally to `bias`. Please use a different name to suppress this warning.
A parameter name that contains `gamma` will be renamed internally to `weight`. Please use a different name to suppress this warning.
A parameter name that contains `beta` will be renamed internally to `bias`. Please use a different name to suppress this warning.
A parameter name that contains `gamma` will be renamed internally to `weight`. Please use a different name to suppress this warning.
A parameter name that contains `beta` will be renamed internally to `bias`. Please use a different name to suppress this warning.
A parameter name that contains `gamma` will be renamed internally to `weight`. Pl

PointerBertModel(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(21128, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSdpaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12, elementw

## 创建评估指标

In [26]:
# 定义评估指标（困惑度Perplexity，越低越好）
import evaluate
accuracy_metric = evaluate.load("accuracy")
f1_metric = evaluate.load("f1")


def compute_metrics(eval_pred):
    # 解析预测结果和标签 (前行输出以及实际标签值)
    predictions, labels = eval_pred
    print(type(predictions), type(labels), type(predictions[0]), type(predictions[0][0]))
    # todo: 自行完成这部分的评估指标的编写
    # 将 logits 转换为预测标签（取最大值索引）
#     predictions = np.argmax(predictions, axis=-1)

#     # 计算指标
#     accuracy = accuracy_metric.compute(predictions=predictions, references=labels)
#     f1 = f1_metric.compute(predictions=predictions, references=labels, average="weighted")  # 多分类用 weighted

#     # 返回指标字典（键为指标名称，值为数值）
#     return {
#         "accuracy": accuracy["accuracy"],
#         "f1": f1["f1"],
#     }
    return {
        "f1": 0.25,
        "accuracy": 0.58
    }

In [19]:

def compute_metrics(eval_pred):
    # 解析预测结果和标签 (前行输出以及实际标签值)
    predictions, labels = eval_pred
    print(type(predictions), type(labels), type(predictions[0]), type(predictions[0][0]))
    # todo: 自行完成这部分的评估指标的编写
    # 将 logits 转换为预测标签（取最大值索引）
#     predictions = np.argmax(predictions, axis=-1)

#     # 计算指标
#     accuracy = accuracy_metric.compute(predictions=predictions, references=labels)
#     f1 = f1_metric.compute(predictions=predictions, references=labels, average="weighted")  # 多分类用 weighted

#     # 返回指标字典（键为指标名称，值为数值）
#     return {
#         "accuracy": accuracy["accuracy"],
#         "f1": f1["f1"],
#     }
    return {
        "f1": 0.25,
        "accuracy": 0.58
    }

## 构造训练参数

In [20]:
from transformers import TrainingArguments, Trainer

# 可能需要安装：pip install transformers[torch]

training_args = TrainingArguments(
    output_dir="./output/bert-finetuned/models",  # 模型保存路径
    overwrite_output_dir=True,
    num_train_epochs=3,  # 训练轮数
    per_device_train_batch_size=4,  # 单设备训练批次大小（视GPU内存调整）
    per_device_eval_batch_size=4,   # 单设备验证批次大小
    gradient_accumulation_steps=4,  # 梯度累积（显存不足时增大，等效于增大batch_size）
    eval_strategy="epoch",    # 每轮结束后验证
    save_strategy="epoch",          # 每轮结束后保存模型
    logging_dir="./output/bert-finetuned/logs",           # 日志路径
    logging_steps=10,
    learning_rate=5e-5,             # 学习率（GPT类模型通常用2e-5 ~ 5e-5）
    weight_decay=0.01,              # 权重衰减（正则化）
    fp16=True,                      # 启用混合精度训练（需GPU支持）
    load_best_model_at_end=True,    # 训练结束后加载最佳模型
    #metric_for_best_model="f1",  # 以准确率为判断标准 默认为损失
    #greater_is_better=True,  # 准确率越高越好（损失则设为 False， 默认为损失）
    eval_do_concat_batches=False # 聚合的数据是否填充合并(False表示不合并) 会影响评估方法
)

## 自定义聚合方法

In [21]:
from transformers.data.data_collator import DataCollatorMixin, pad_without_fast_tokenizer_warning
from transformers import PreTrainedTokenizerBase
from torch import Tensor
from typing import List, Any, Dict

@dataclass
class MyDataCollator(DataCollatorMixin):
    tokenizer: PreTrainedTokenizerBase
    
    def _expand_values(self, values, constant_values=-100) -> Tensor:
        """
        仅考虑对numpy数据进行处理
        """
        values = [np.asarray(value) for value in values]

        value_shapes = [np.shape(value) for value in values]
        target_shape = np.max(value_shapes, axis=0)
        new_values = []
        for j in range(len(values)):
            # 处理第j个样本
            pad_width = []
            for i in range(len(target_shape)):
                target_i = target_shape[i] # 第i维的期望大小
                ori_j_i = value_shapes[j][i] # 第j个样本的第i维的原始大小
                pad_width.append((0, target_i - ori_j_i))
            # 填充样本
            new_values.append(
                np.pad(values[j], pad_width=pad_width,  mode='constant', constant_values=constant_values)
            )

        new_values = np.stack(new_values) # 批次合并

        return torch.tensor(new_values)

    def __call__(self, features: List[Dict[str, Any]]) -> Dict[str, Any]:
        
        # dict_keys(['text', 'input_ids', 'attention_mask', 'sub_labels', 'sub_masks', 'obj_labels'])

        
        # list -> dict
        features = {key: [feature[key] for feature in features] for key in features[0].keys()}
        
        # 对input_ids和attention_masks进行转换
        batch = pad_without_fast_tokenizer_warning(
            self.tokenizer,
            {
                'input_ids': features['input_ids'],
                'attention_mask': features['attention_mask'],
            },
            padding=True,
            max_length=None,
            pad_to_multiple_of=None,
            return_tensors="pt",
        )

        # 对其它标签进行填充
        for key in ['sub_labels', 'sub_masks', 'obj_labels']:
            batch[key] = self._expand_values(features[key], constant_values=-100 if key != 'sub_masks' else 0)

        return batch

## 迭代训练

In [22]:
from transformers import DataCollatorForTokenClassification, DataCollatorWithPadding

# 数据填充对象
data_collator = MyDataCollator(tokenizer=tokenizer)

# 初始化Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    data_collator=data_collator,  # 传入数据整理器
    train_dataset=dataset["train"],
    eval_dataset=dataset["test"],
    compute_metrics=compute_metrics,  # 评估指标
)

# 开始训练
trainer.train()

Epoch,Training Loss,Validation Loss,F1,Accuracy
1,No log,3.339123,0.250000,0.580000
2,No log,3.293667,0.250000,0.580000


<class 'list'> <class 'list'> <class 'tuple'> <class 'numpy.ndarray'>
<class 'list'> <class 'list'> <class 'tuple'> <class 'numpy.ndarray'>


TrainOutput(global_step=3, training_loss=1.7031981150309246, metrics={'train_runtime': 26.8932, 'train_samples_per_second': 1.004, 'train_steps_per_second': 0.112, 'total_flos': 885276066720.0, 'train_loss': 1.7031981150309246, 'epoch': 2.0})

In [23]:
# 模型保存(最终的最优模型保存)
trainer.save_model(os.path.join(training_args.output_dir, "./final-best-model"))

# 模型推理应用